In [1]:
import optuna
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from datetime import datetime
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import roc_auc_score
import numpy as np

Model 1:

In [2]:
def objective(trial, features, target):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 1000),
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 300),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = cross_val_score(model, features, target, cv=cv, scoring='roc_auc')
    return auc_scores.mean()


In [3]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_1"
    model_category = "rf"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 50 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2013))
    study.optimize(lambda trial: objective(trial, X_data, y_data), n_trials=50, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 03:40:00,880] A new study created in memory with name: model_1_tuning
[I 2026-05-11 03:40:15,093] Trial 5 finished with value: 0.8848260661198848 and parameters: {'n_estimators': 56, 'max_depth': 29, 'criterion': 'gini', 'min_samples_split': 29, 'min_samples_leaf': 108, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 5 with value: 0.8848260661198848.
[I 2026-05-11 03:40:40,610] Trial 4 finished with value: 0.8802317623307477 and parameters: {'n_estimators': 106, 'max_depth': 11, 'criterion': 'entropy', 'min_samples_split': 275, 'min_samples_leaf': 141, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 5 with value: 0.8848260661198848.
[I 2026-05-11 03:40:42,893] Trial 15 finished with value: 0.8605497005830379 and parameters: {'n_estimators': 215, 'max_depth': 2, 'criterion': 'gini', 'min_samples_split': 140, 'min_samples_leaf': 111, 'max_features': 'log2', 'class_weight': 'balanced'}. Best is trial 5 with value: 0.8848260661198848.

Time: 0:11:31.769160


Model 2:

In [4]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_2"
    model_category = "rf"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 50 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2013))
    study.optimize(lambda trial: objective(trial, X_data, y_data), n_trials=50, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 03:51:32,674] A new study created in memory with name: model_2_tuning
[I 2026-05-11 03:51:48,538] Trial 15 finished with value: 0.8051652580738511 and parameters: {'n_estimators': 61, 'max_depth': 27, 'criterion': 'entropy', 'min_samples_split': 35, 'min_samples_leaf': 172, 'max_features': 'log2', 'class_weight': 'balanced'}. Best is trial 15 with value: 0.8051652580738511.
[I 2026-05-11 03:51:56,657] Trial 9 finished with value: 0.7941209751365467 and parameters: {'n_estimators': 113, 'max_depth': 3, 'criterion': 'gini', 'min_samples_split': 188, 'min_samples_leaf': 55, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 15 with value: 0.8051652580738511.
[I 2026-05-11 03:52:29,130] Trial 17 finished with value: 0.801520653970781 and parameters: {'n_estimators': 103, 'max_depth': 6, 'criterion': 'entropy', 'min_samples_split': 156, 'min_samples_leaf': 118, 'max_features': None, 'class_weight': None}. Best is trial 15 with value: 0.8051652580738511.
[I 2026

Time: 0:17:52.618405


Model 1a:

In [5]:
def objective_a(trial, features, target):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 1000),
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 300),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = []
    # cross validation loop for oversample
    for train_idx, val_idx in cv.split(features, target):
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target[train_idx], target[val_idx]

        # oversample
        oversampler = RandomOverSampler()
        X_train_resampled, y_train_resampled = oversampler.fit_resample(X_train, y_train)

        # use oversample data to train model
        model.fit(X_train_resampled, y_train_resampled)
        
        # calculate auc
        y_proba = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, y_proba)
        auc_scores.append(fold_auc)

    # return 5-fold cv auc
    return np.mean(auc_scores)

In [6]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_1a"
    model_category = "rf"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 50 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2013))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=50, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 04:09:25,290] A new study created in memory with name: model_1a_tuning
[I 2026-05-11 04:09:53,279] Trial 0 finished with value: 0.7520480832782551 and parameters: {'n_estimators': 136, 'max_depth': 2, 'criterion': 'gini', 'min_samples_split': 26, 'min_samples_leaf': 49, 'max_features': None, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7520480832782551.
[I 2026-05-11 04:10:09,731] Trial 19 finished with value: 0.8374496034368804 and parameters: {'n_estimators': 100, 'max_depth': 13, 'criterion': 'gini', 'min_samples_split': 242, 'min_samples_leaf': 45, 'max_features': None, 'class_weight': 'balanced_subsample'}. Best is trial 19 with value: 0.8374496034368804.
[I 2026-05-11 04:10:11,319] Trial 9 finished with value: 0.822100866563854 and parameters: {'n_estimators': 289, 'max_depth': 3, 'criterion': 'entropy', 'min_samples_split': 254, 'min_samples_leaf': 137, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 19 with value: 0.837449603436880

Time: 0:07:18.861218


Model 2a:

In [7]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_2a"
    model_category = "rf"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 50 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2013))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=50, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 04:16:44,168] A new study created in memory with name: model_2a_tuning
[I 2026-05-11 04:17:00,014] Trial 2 finished with value: 0.7343383167608656 and parameters: {'n_estimators': 81, 'max_depth': 3, 'criterion': 'gini', 'min_samples_split': 101, 'min_samples_leaf': 117, 'max_features': None, 'class_weight': None}. Best is trial 2 with value: 0.7343383167608656.
[I 2026-05-11 04:17:04,262] Trial 3 finished with value: 0.745810554955612 and parameters: {'n_estimators': 140, 'max_depth': 2, 'criterion': 'entropy', 'min_samples_split': 181, 'min_samples_leaf': 39, 'max_features': 'log2', 'class_weight': None}. Best is trial 3 with value: 0.745810554955612.
[I 2026-05-11 04:17:07,933] Trial 15 finished with value: 0.7332819095970717 and parameters: {'n_estimators': 127, 'max_depth': 3, 'criterion': 'entropy', 'min_samples_split': 160, 'min_samples_leaf': 78, 'max_features': None, 'class_weight': 'balanced'}. Best is trial 3 with value: 0.745810554955612.
[I 2026-05-11 04:17:1

Time: 0:05:23.945501


Model 1b:

In [8]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_1b"
    model_category = "rf"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 50 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2013))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=50, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 04:22:08,144] A new study created in memory with name: model_1b_tuning
[I 2026-05-11 04:23:11,055] Trial 12 finished with value: 0.8798881180364733 and parameters: {'n_estimators': 115, 'max_depth': 19, 'criterion': 'gini', 'min_samples_split': 271, 'min_samples_leaf': 27, 'max_features': None, 'class_weight': None}. Best is trial 12 with value: 0.8798881180364733.
[I 2026-05-11 04:23:18,250] Trial 18 finished with value: 0.8739453978775075 and parameters: {'n_estimators': 136, 'max_depth': 15, 'criterion': 'gini', 'min_samples_split': 282, 'min_samples_leaf': 108, 'max_features': None, 'class_weight': None}. Best is trial 12 with value: 0.8798881180364733.
[I 2026-05-11 04:23:48,081] Trial 0 finished with value: 0.8554105059202683 and parameters: {'n_estimators': 444, 'max_depth': 5, 'criterion': 'gini', 'min_samples_split': 116, 'min_samples_leaf': 102, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 12 with value: 0.8798881180364733.
[I 2026-05-11 04:24:00

Time: 0:10:17.281684


Model 2b:

In [9]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_2b"
    model_category = "rf"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 50 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2013))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=50, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 04:32:25,441] A new study created in memory with name: model_2b_tuning
[I 2026-05-11 04:32:57,359] Trial 12 finished with value: 0.8054935540265046 and parameters: {'n_estimators': 72, 'max_depth': 5, 'criterion': 'entropy', 'min_samples_split': 112, 'min_samples_leaf': 96, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}. Best is trial 12 with value: 0.8054935540265046.
[I 2026-05-11 04:32:58,734] Trial 10 finished with value: 0.7853589174661708 and parameters: {'n_estimators': 144, 'max_depth': 2, 'criterion': 'gini', 'min_samples_split': 152, 'min_samples_leaf': 181, 'max_features': 'log2', 'class_weight': None}. Best is trial 12 with value: 0.8054935540265046.
[I 2026-05-11 04:33:16,828] Trial 8 finished with value: 0.7964108253382918 and parameters: {'n_estimators': 219, 'max_depth': 3, 'criterion': 'entropy', 'min_samples_split': 140, 'min_samples_leaf': 56, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 12 with value: 0.8054935540265046.


Time: 0:22:20.783350
